# GW150914 with pyEFPEHM, and an SMBHB memory example

This notebook exercises the public interfaces on
`feature/tdi-single-link-pytdi-probe`:

- `PyEFPEHMSource` supplies carrier-factored time-domain harmonics at
  arbitrary retarded times;
- PyTDI supplies the verified TDI-2 delay polynomials, while
  `adaptive_sparse_tdi_response` applies the moving light-cone and
  first-order-velocity single-link response in the time domain;
- the slow response brackets are evaluated sparsely, reconstructed in
  bounded-memory chunks, FFTed, and passed to standard `pycbc.filter.sigma`;
- `SpaceDetector("LISA", backend="pycbc")` remains the dense reference path
  for the short NRHybSur3dq8_CCE merger signal.

The pyEFPEHM example follows a GW150914-like stellar-mass binary for two
years before 0.1 Hz. The CCE example is a separate
$10^6+5\times10^5\,M_\odot$ source-frame SMBHB at $z=1$. Both sections are
enabled by default; set `RUN_CCE=0` only when the surrogate or its optional
dependencies are deliberately unavailable.


## Installation

```bash
python3.11 -m venv ~/.venvs/pycbc-tdi-tutorial
source ~/.venvs/pycbc-tdi-tutorial/bin/activate
python -m pip install --upgrade pip wheel

git clone --branch feature/tdi-single-link-pytdi-probe \
  https://github.com/WuShichao/pycbc.git pycbc-tdi-probe
cd pycbc-tdi-probe
python -m pip install -e .
python -m pip install "pytdi==2.2.1" matplotlib jupyter ipykernel

git clone --branch feature/pyefpehm-pycbc-mode-integration \
  https://github.com/WuShichao/pyEFPEHM.git ../pyEFPEHM
python -m pip install -e ../pyEFPEHM

python -m ipykernel install --user \
  --name pycbc-tdi-tutorial --display-name "PyCBC TDI tutorial"
```

The CCE/memory section is enabled by default and additionally needs:

```bash
python -m pip install "gwsurrogate==1.2.0" "foutstep==0.1.4" h5py
export NRHYBSUR_CCE_PATH=/absolute/path/to/NRHybSur3dq8_CCE.h5
```

In [1]:
import gc
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

from pycbc.coordinates.space_orbit import LisaEqualArmOrbit
from pycbc.detector.space import SpaceDetector
from pycbc.fft import fft
from pycbc.filter import sigma
from pycbc.psd.analytical_space import (
    analytical_psd_lisa_tdi_AE,
    analytical_psd_lisa_tdi_T,
    sensitivity_curve_lisa_confusion,
)
from pycbc.tdi import (
    PyEFPEHMSource,
    TimeShiftedHarmonicSource,
    adaptive_sparse_tdi_response,
    delay_padding,
    frequency_polarization_series,
    orthogonal_channels,
)
from pycbc.tdi.backends.pytdi_backend import (
    PyTDICombinationAdapter,
    get_pytdi_combination,
)
from pycbc.tdi.sources import ArrayWaveformSource
from pycbc.types import FrequencySeries, TimeSeries

YEAR = 365.25 * 86400.0
DAY = 86400.0
C_SI = 299792458.0
ARM_LENGTH = 2.5e9
DT = 5.0
PYEFPE_DT = float(os.environ.get("PYEFPE_TDI_DELTA_T", "4.0"))
TIME_CHUNK_SIZE = int(os.environ.get("TDI_TIME_CHUNK_SIZE", "262144"))
TDI_GENERATION = 2
TDI_PSD_VERSION = "2.0"
FULL_DURATION = 2 * YEAR
QUICK = os.environ.get("TDI_NOTEBOOK_QUICK", "0") == "1"
RUN_PYEFPEHM = os.environ.get("RUN_PYEFPEHM", "1") == "1"
RUN_CCE = os.environ.get("RUN_CCE", "1") == "1"
DURATION = 7 * DAY if QUICK else FULL_DURATION
FD_DELTA_F = float(os.environ.get("TDI_NOTEBOOK_DELTA_F", "5e-6"))
RESPONSE_TOLERANCE = float(os.environ.get("TDI_RESPONSE_TOLERANCE", "1e-3"))
TDI_NULL_SINE_MIN = 1e-3
CACHE = Path(os.environ.get(
    "TDI_NOTEBOOK_CACHE", "/tmp/gw150914-tdi-notebook"
))
CACHE.mkdir(parents=True, exist_ok=True)

if sys.prefix.rstrip("/") == "/home/nightwing/anaconda3":
    raise RuntimeError("Select the dedicated kernel; Conda base is unsupported")

print("interpreter:", sys.executable)
print("mode:", "7-day smoke test" if QUICK else "two-year run")
print("sections:", {"pyEFPEHM": RUN_PYEFPEHM, "CCE": RUN_CCE})
print("frequency spacing:", FD_DELTA_F, "Hz")
print("cache:", CACHE)


/home/nightwing/anaconda3/envs/tutorial/lib/python3.11/importlib/__init__.py:126: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  return _bootstrap._gcd_import(name[level:], package, level)
PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1


interpreter: /home/nightwing/anaconda3/envs/tutorial/bin/python
mode: two-year run
sections: {'pyEFPEHM': True, 'CCE': False}
frequency spacing: 5e-06 Hz
cache: /tmp/gw150914-tdi-notebook


## Source and detector choices

GW150914 masses, redshift and distance use the GWOSC summary values.
Detector-frame masses are $(1+z)m_i$. Inclination, phase and ecliptic sky
coordinates are fixed reproducible choices, not posterior estimates.

In [2]:
SOURCE_M1, SOURCE_M2 = 36.2, 29.1
REDSHIFT, DISTANCE = 0.09, 420.0
MASS1, MASS2 = (1 + REDSHIFT) * np.array([SOURCE_M1, SOURCE_M2])
INCLINATION, PHASE = 1.0, 0.0
LAMBDA, BETA = 0.9, -0.25

# The generation band deliberately extends past the two-year analysis band.
# This gives delayed TDI terms valid source data at both observation edges.
F22_GENERATION_START = 0.0209846
F22_ANALYSIS_END = 0.1
F22_GENERATION_END = 0.1005
pyefpe_parameters = dict(
    mass1=MASS1,
    mass2=MASS2,
    distance=DISTANCE,
    eccentricity=0.0,
    spin1z=0.0,
    spin2z=0.0,
    inclination=INCLINATION,
    phase=PHASE,
    f22_start=F22_GENERATION_START,
    f22_ref=F22_GENERATION_START,
    f22_end=F22_GENERATION_END,
    Amplitude_tol=1e-4,
)

cce_parameters = dict(
    source_mass1=1.0e6,
    source_mass2=5.0e5,
    redshift=1.0,
    distance_mpc=6800.0,
)
{"GW150914-like": pyefpe_parameters, "CCE SMBHB": cce_parameters}


{'GW150914-like': {'mass1': np.float64(39.458000000000006),
  'mass2': np.float64(31.719000000000005),
  'distance': 420.0,
  'eccentricity': 0.0,
  'spin1z': 0.0,
  'spin2z': 0.0,
  'inclination': 1.0,
  'phase': 0.0,
  'f22_start': 0.0209846,
  'f22_ref': 0.0209846,
  'f22_end': 0.1005,
  'Amplitude_tol': 0.0001},
 'CCE SMBHB': {'source_mass1': 1000000.0,
  'source_mass2': 500000.0,
  'redshift': 1.0,
  'distance_mpc': 6800.0}}

## Standard SNR, PSD and plotting helpers

There is no special fast SNR statistic. The sky-averaged reference keeps
pyEFPEHM's native FD waveform, but the explicit A/E/T signal is reconstructed
from the time-domain on-the-fly response and transformed with an ordinary
FFT. Both are standard `FrequencySeries` inputs to `sigma`, which performs
PyCBC's $4\Delta f$ noise-weighted norm. `PYEFPE_TDI_DELTA_T` controls the
time-domain Nyquist frequency; `FD_DELTA_F` only controls the independent
sky-averaged reference quadrature.

The averaged curve is a strain PSD paired with $h_+,h_\times$. The A/E/T
curves use the equal-static-arm analytic TDI-2 PSD while the signal uses a
moving constellation. Static PSD zeros absent from that moving response are
masked before the standard `sigma` call, so `rho_full_tdi` is a diagnostic
until noise is propagated through the same dynamic delay operators.

`fourier_series` is shared by the pyEFPEHM and CCE time-domain paths.


In [3]:
def disk_array(label, size):
    values = np.memmap(
        CACHE / f"{label}.f64", dtype="float64", mode="w+", shape=size
    )
    values[:] = 0.0
    return values


def fourier_series(values, delta_t=DT, taper=True):
    """Transform a time-domain response with PyCBC's FFT backend."""
    data = np.asarray(values).copy()
    if taper:
        nedge = min(round(DAY / delta_t), len(data) // 2)
        if nedge:
            ramp = 0.5 * (1 - np.cos(np.pi * np.arange(nedge) / nedge))
            data[:nedge] *= ramp
            data[-nedge:] *= ramp[::-1]
    series = TimeSeries(data, delta_t=delta_t, copy=False)
    transformed = FrequencySeries(
        np.empty(len(data) // 2 + 1, dtype=np.complex128),
        delta_f=1.0 / (len(data) * delta_t),
        copy=False,
    )
    fft(series, transformed)
    return transformed


def lisa_psd(kind, length, delta_f, years):
    if kind == "sky":
        return sensitivity_curve_lisa_confusion(
            length, delta_f, delta_f,
            base_model="SciRD", duration=years,
        )
    function = (
        analytical_psd_lisa_tdi_AE
        if kind == "AE" else analytical_psd_lisa_tdi_T
    )
    return function(length, delta_f, delta_f, tdi=TDI_PSD_VERSION)


def sigma_with_valid_psd(series, psd, f_low, f_high=0.1, tdi=False):
    """Call standard PyCBC sigma after excluding invalid analytic-PSD bins."""
    signal_values = np.asarray(series)
    noise_values = np.asarray(psd)
    frequency = np.asarray(series.sample_frequencies)
    good = np.isfinite(noise_values) & (noise_values > 0)
    if tdi:
        x = 2 * np.pi * frequency * ARM_LENGTH / C_SI
        good &= np.abs(np.sin(x)) > TDI_NULL_SINE_MIN
        if TDI_GENERATION == 2:
            good &= np.abs(np.sin(2 * x)) > TDI_NULL_SINE_MIN
    if not np.all(good):
        # Both arrays are private products of this analysis step. Masking
        # in place avoids two observation-length copies before sigma.
        signal_values[~good] = 0.0
        noise_values[~good] = np.inf
    return float(sigma(
        series,
        psd=psd,
        low_frequency_cutoff=f_low,
        high_frequency_cutoff=f_high,
    ))


def log_curve(frequency, values, low, high, bins=900, envelope=False):
    target = np.geomspace(low, high, bins)
    index = np.clip(np.searchsorted(frequency, target), 0, len(frequency) - 1)
    if not envelope:
        return frequency[index], values[index]
    edges = np.searchsorted(frequency, np.geomspace(low, high, bins + 1))
    chosen = [
        a + np.argmax(values[a:b])
        for a, b in zip(edges[:-1], edges[1:], strict=True) if b > a
    ]
    return frequency[chosen], values[chosen]


def summarize_frequency_series(hp, hc, aet, duration, f_low, f_high=0.1):
    """Use standard sigma on already-generated PyCBC FD waveforms."""
    years = duration / YEAR
    frequency = np.asarray(hp.sample_frequencies)
    sky_psd = lisa_psd("sky", len(hp), hp.delta_f, years)
    rho_source2 = sum(
        sigma_with_valid_psd(series, sky_psd, f_low, f_high) ** 2
        for series in (hp, hc)
    )
    source_signal = 2 * frequency * np.sqrt(
        np.abs(np.asarray(hp)) ** 2 + np.abs(np.asarray(hc)) ** 2
    )
    source_noise = np.sqrt(frequency * np.asarray(sky_psd))
    curves = {"source": (
        *log_curve(frequency, source_signal, f_low, f_high, bins=6000, envelope=True),
        *log_curve(frequency, source_noise, f_low, f_high, bins=6000),
    )}
    snr = {"rho_sky_average": float(np.sqrt(rho_source2))}
    aet_items = aet.items() if hasattr(aet, "items") else aet
    for name, series in aet_items:
        psd = lisa_psd("AE" if name in "AE" else "T", len(series), series.delta_f, years)
        snr[f"rho_{name}"] = sigma_with_valid_psd(
            series, psd, f_low, f_high, tdi=True
        )
        channel_frequency = np.asarray(series.sample_frequencies)
        signal = 2 * channel_frequency * np.abs(np.asarray(series))
        noise = np.sqrt(channel_frequency * np.asarray(psd))
        curves[name] = (
            *log_curve(channel_frequency, signal, f_low, f_high, bins=6000, envelope=True),
            *log_curve(channel_frequency, noise, f_low, f_high, bins=6000),
        )
    snr["rho_full_tdi"] = float(np.sqrt(sum(
        snr[f"rho_{name}"] ** 2 for name in "AET"
    )))
    return curves, snr


def summarize_dense(source_channels, duration=DURATION):
    """Memory-bounded CCE adapter; only one large FFT lives at a time."""
    years = duration / YEAR
    frequency = None
    source_power = None
    rho_source2 = 0.0
    curves = {}
    for name in ("hp", "hc"):
        series = fourier_series(source_channels[name])
        if frequency is None:
            frequency = np.asarray(series.sample_frequencies)
            source_power = np.zeros(len(series))
        psd = lisa_psd("sky", len(series), series.delta_f, years)
        rho_source2 += sigma_with_valid_psd(
            series, psd, 1e-5, F22_ANALYSIS_END
        ) ** 2
        source_power += np.abs(np.asarray(series)) ** 2
        del series, psd
    sky_psd = lisa_psd("sky", len(frequency), frequency[1], years)
    source_signal = 2 * frequency * np.sqrt(source_power)
    source_noise = np.sqrt(frequency * np.asarray(sky_psd))
    curves["source"] = (
        *log_curve(frequency, source_signal, 1e-5, 0.1, bins=6000, envelope=True),
        *log_curve(frequency, source_noise, 1e-5, 0.1, bins=6000),
    )
    del source_power, source_signal, source_noise, sky_psd

    snr = {"rho_sky_average": float(np.sqrt(rho_source2))}
    for name in "AET":
        series = fourier_series(source_channels[name])
        psd = lisa_psd(
            "AE" if name in "AE" else "T",
            len(series), series.delta_f, years,
        )
        snr[f"rho_{name}"] = sigma_with_valid_psd(
            series, psd, 1e-5, 0.1, tdi=True
        )
        signal = 2 * frequency * np.abs(np.asarray(series))
        noise = np.sqrt(frequency * np.asarray(psd))
        curves[name] = (
            *log_curve(frequency, signal, 1e-5, 0.1, bins=6000, envelope=True),
            *log_curve(frequency, noise, 1e-5, 0.1, bins=6000),
        )
        del series, psd, signal, noise
    snr["rho_full_tdi"] = float(np.sqrt(sum(
        snr[f"rho_{name}"] ** 2 for name in "AET"
    )))
    return curves, snr


def plot_lisa_bands(curves, snr, title, bands):
    fig, axes = plt.subplots(2, len(bands), figsize=(5 * len(bands), 7), sharey="row")
    for column, (low, high) in enumerate(bands):
        signal_f, signal, noise_f, noise = curves["source"]
        selected = (signal_f >= low) & (signal_f <= high)
        axes[0, column].loglog(signal_f[selected], signal[selected], label="source")
        selected = (noise_f >= low) & (noise_f <= high)
        axes[0, column].loglog(noise_f[selected], noise[selected], label="sky-average noise")
        for name in "AET":
            signal_f, signal, noise_f, noise = curves[name]
            selected = (signal_f >= low) & (signal_f <= high)
            axes[1, column].loglog(signal_f[selected], signal[selected], label=name)
            selected = (noise_f >= low) & (noise_f <= high)
            axes[1, column].loglog(noise_f[selected], noise[selected], ls="--", alpha=0.55)
        for row in range(2):
            axes[row, column].set_xlim(low, high)
            axes[row, column].grid(which="both", alpha=0.25)
            axes[row, column].set_xlabel("frequency [Hz]")
        axes[0, column].set_title(f"{low:g}–{high:g} Hz")
    axes[0, 0].set_ylabel(r"$2f|\tilde h|$ or $\sqrt{fS_h}$")
    axes[1, 0].set_ylabel(r"$2f|\widetilde{\mathrm{TDI}}|$ or $\sqrt{fS_n}$")
    axes[0, -1].legend(fontsize=8)
    axes[1, -1].legend(fontsize=8)
    fig.suptitle(
        f"{title}: sky-average SNR={snr['rho_sky_average']:.3g}, "
        f"full-TDI SNR={snr['rho_full_tdi']:.3g}"
    )
    plt.show()


## pyEFPEHM: native FD waveform and direct TDI-2 response

`TimeShiftedHarmonicSource` maps the selected observation interval onto
mission time $[0,T_{\rm obs}]$ while retaining the waveform history needed
by retarded link queries. The detector response does **not** use a frozen
$t(f)$ or a Fourier-domain transfer approximation. PyTDI supplies the
verified TDI-2 delay polynomials; PyCBC evaluates their retarded waveform
arguments through the carrier-factored time-domain on-the-fly response.

Only the slowly varying response brackets are built on adaptive sparse
grids. They are then reconstructed in bounded-memory chunks at the cadence
required by the highest analysed frequency, transformed with an ordinary
FFT, and passed to standard PyCBC `sigma`. The sparse plot is selected from
that same time series; it is no longer a different response route.


In [4]:
def tdi_terms(name):
    combination = get_pytdi_combination(name)
    return PyTDICombinationAdapter(name, combination, delta_t=DT).terms()


def build_pyefpe_source(parameters, duration=DURATION):
    native = PyEFPEHMSource(parameters)
    reference_harmonic = (2, 2, 2)

    def f22(source_time):
        return float(
            native.angular_frequency(reference_harmonic, source_time)
            / (2 * np.pi)
        )

    analysis_end = brentq(
        lambda source_time: f22(source_time) - F22_ANALYSIS_END,
        native.t_start,
        native.t_end,
    )
    analysis_start = analysis_end - duration
    if analysis_start < native.t_start:
        raise ValueError(
            "pyEFPEHM generation starts after the requested observation"
        )
    # Keep the native generation margins. TDI link and operator delays
    # query just outside the recorded [0, duration) interval.
    source = TimeShiftedHarmonicSource(native, offset=analysis_start)
    return source, f22(analysis_start)


def run_pyefpe(parameters, duration=DURATION):
    timings = {}
    started = time.perf_counter()
    source, f_low = build_pyefpe_source(parameters, duration)
    timings["source initialization"] = time.perf_counter() - started

    started = time.perf_counter()
    hp, hc = frequency_polarization_series(
        source,
        delta_f=FD_DELTA_F,
        f_lower=f_low,
        f_final=F22_ANALYSIS_END,
    )
    timings["native FD polarizations"] = time.perf_counter() - started

    orbit = LisaEqualArmOrbit(t0=0.0)
    terms = {
        name: tdi_terms(f"{name}{TDI_GENERATION}") for name in "XYZ"
    }
    all_terms = tuple(
        term for channel_terms in terms.values() for term in channel_terms
    )
    padding = delay_padding(
        orbit, np.array([0.0, duration]), all_terms
    )
    if source.t_start > -padding or source.t_end < duration + padding:
        raise ValueError(
            "pyEFPEHM generation band does not cover the retarded TDI "
            f"queries: need [{-padding}, {duration + padding}], got "
            f"[{source.t_start}, {source.t_end}]"
        )

    started = time.perf_counter()
    sparse = adaptive_sparse_tdi_response(
        source,
        orbit,
        terms,
        LAMBDA,
        BETA,
        t_start=0.0,
        t_end=duration,
        initial_step=DAY,
        relative_tolerance=RESPONSE_TOLERANCE,
        velocity_order=1,
    )
    timings["sparse time-domain XYZ/TDI-2 response"] = (
        time.perf_counter() - started
    )

    aet_matrix = np.stack(orthogonal_channels(*np.eye(3)))
    aet_response = sparse.linear_transform(aet_matrix, tuple("AET"))
    sample_count = int(np.ceil(
        np.nextafter(float(duration), 0.0) / PYEFPE_DT
    ))
    storage = {
        name: disk_array(f"pyefpe_tdi2_{name}", sample_count)
        for name in "AET"
    }
    started = time.perf_counter()
    aet_timeseries = aet_response.to_timeseries(
        PYEFPE_DT,
        t_start=0.0,
        t_end=duration,
        channels=tuple("AET"),
        chunk_size=TIME_CHUNK_SIZE,
        out=storage,
    )
    timings["chunked uniform time-domain reconstruction"] = (
        time.perf_counter() - started
    )

    def aet_frequency_series():
        for name in "AET":
            yield name, fourier_series(
                aet_timeseries[name], delta_t=PYEFPE_DT
            )

    started = time.perf_counter()
    curves, snr = summarize_frequency_series(
        hp, hc, aet_frequency_series(), duration,
        f_low=f_low, f_high=F22_ANALYSIS_END
    )
    timings["time-domain FFT + standard PyCBC sigma"] = (
        time.perf_counter() - started
    )

    plot_index = np.unique(np.linspace(
        0, sample_count - 1, min(12001, sample_count), dtype=int
    ))
    plot_times = plot_index * PYEFPE_DT
    aet_time = {
        name: np.asarray(aet_timeseries[name])[plot_index]
        for name in "AET"
    }
    hp_time, hc_time = source.polarizations(plot_times)

    return {
        "source": source,
        "f_low": f_low,
        "hp": hp,
        "hc": hc,
        "aet_timeseries": aet_timeseries,
        "time": plot_times,
        "hp_time": hp_time,
        "hc_time": hc_time,
        "aet_time": aet_time,
        "curves": curves,
        "snr": snr,
        "time_diagnostics": sparse.diagnostics,
        "timings": timings,
    }


In [5]:
if RUN_PYEFPEHM:
    pyefpe = run_pyefpe(pyefpe_parameters)
    print("analysis band:", pyefpe["f_low"], "to", F22_ANALYSIS_END, "Hz")
    print("time-domain response grids:", pyefpe["time_diagnostics"])
    print("timings [s]:", pyefpe["timings"])
    print("SNR:", pyefpe["snr"])

    years = pyefpe["time"] / YEAR
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(years, pyefpe["hp_time"], lw=0.7, label=r"$h_+$")
    axes[0].plot(years, pyefpe["hc_time"], lw=0.7, label=r"$h_\times$")
    axes[1].plot(years, pyefpe["aet_time"]["A"], lw=0.7, label="TDI-2.0 A")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    axes[1].set_xlabel("mission time [yr]")
    plt.show()

    plot_lisa_bands(
        pyefpe["curves"], pyefpe["snr"], "GW150914-like pyEFPEHM",
        [(0.018, 0.03), (0.03, 0.06), (0.06, 0.1)],
    )


/mnt/d/pycbc-tdi-probe/pycbc/psd/read.py:76: RuntimeWarning: divide by zero encountered in log
  slog = numpy.log(noise_data)
/tmp/ipykernel_22925/1961122667.py:90: RuntimeWarning: invalid value encountered in multiply
  source_noise = np.sqrt(frequency * np.asarray(sky_psd))


/tmp/ipykernel_22925/1961122667.py:104: RuntimeWarning: invalid value encountered in multiply
  noise = np.sqrt(channel_frequency * np.asarray(psd))


analysis band: 0.020985080534504003 to 0.1 Hz
time-domain response grids: {(np.int64(2), np.int64(2), np.int64(2)): {'grid_points': 2646, 'tested_points': 9849, 'deepest_refinement': 3, 'relative_tolerance': 0.001}}
timings [s]: {'source initialization': 0.6213686179999058, 'native FD polarizations': 0.015123608999601856, 'sparse time-domain XYZ/TDI-2 response': 2.3228650979999657, 'chunked uniform time-domain reconstruction': 3.1008192449999115, 'time-domain FFT + standard PyCBC sigma': 11.02571261399953}
SNR: {'rho_sky_average': 3.1473169906405682, 'rho_A': 2.6034245294789042, 'rho_E': 2.61176190779393, 'rho_T': 0.7417853757576525, 'rho_full_tdi': 3.7615641809470386}


/tmp/ipykernel_22925/3139270602.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_22925/1961122667.py:192: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## NRHybSur3dq8_CCE SMBHB: memory from PTA to LISA

This section calls `gwsurrogate` for the waveform and the public
`SpaceDetector.project_wave` entry point for PyCBC's complete dense
response. Chunking is retained because one two-year, 5 s array has
12.6 million samples; it is resource control, not a second response
implementation. The oscillatory part is tapered below Nyquist while the
permanent $m=0$ memory plateau is kept.

The PTA panel is a strain-spectrum diagnostic, not a PTA SNR. A PTA SNR
additionally needs pulsar locations, cadence, Earth/pulsar terms, red noise
and timing-model projection.

In [6]:
CCE_MEMORY_MODES = ((2, 0), (3, 0), (4, 0))
CCE_MODES = ((2, 2), *CCE_MEMORY_MODES)


def cce_waveform(model, parameters):
    m1 = parameters["source_mass1"] * (1 + parameters["redshift"])
    m2 = parameters["source_mass2"] * (1 + parameters["redshift"])
    common = dict(
        q=m1 / m2,
        chiA0=[0, 0, 0],
        chiB0=[0, 0, 0],
        M=m1 + m2,
        dist_mpc=parameters["distance_mpc"],
        f_low=0.0,
        f_ref=None,
        dt=None,
        units="mks",
    )
    times, modes = model(**common, mode_list=CCE_MODES)[:2]
    total = model._mode_sum(modes, INCLINATION, np.pi / 2 - PHASE, fake_neg_modes=False)
    memory_modes = {mode: modes[mode] for mode in CCE_MEMORY_MODES}
    memory = model._mode_sum(
        memory_modes, INCLINATION, np.pi / 2 - PHASE, fake_neg_modes=False
    )
    memory -= memory[0]

    oscillatory = total - memory
    phase22 = np.unwrap(np.angle(modes[(2, 2)]))
    frequency22 = np.maximum.accumulate(
        np.abs(np.gradient(phase22, times)) / (2 * np.pi)
    )
    taper = np.ones_like(times)
    transition = (frequency22 > 0.07) & (frequency22 < 0.09)
    taper[frequency22 >= 0.09] = 0.0
    taper[transition] = 0.5 * (
        1 + np.cos(np.pi * (frequency22[transition] - 0.07) / 0.02)
    )
    return times, memory + taper * oscillatory, memory


def mission_source(times, strain, memory, duration=DURATION, guard=4096.0):
    times = np.asarray(times, dtype=float)
    strain = np.asarray(strain, dtype=complex)
    memory = np.asarray(memory, dtype=complex)
    if (times.ndim != 1 or strain.shape != times.shape
            or memory.shape != times.shape):
        raise ValueError(
            "CCE times, strain and memory must be matching one-dimensional arrays"
        )
    if len(times) < 2 or np.any(~np.isfinite(times)) or np.any(np.diff(times) <= 0):
        raise ValueError("CCE times must contain at least two finite increasing samples")

    event_time = duration / 2
    raw_times = times + event_time
    window_start = -float(guard)
    window_stop = float(duration) + float(guard)
    inside = (raw_times > window_start) & (raw_times < window_stop)

    left_strain = (
        np.interp(window_start, raw_times, strain)
        if window_start <= raw_times[-1]
        else memory[-1]
    )
    right_strain = (
        np.interp(window_stop, raw_times, strain)
        if window_stop <= raw_times[-1]
        else memory[-1]
    )
    sample_times = np.concatenate(([window_start], raw_times[inside], [window_stop]))
    windowed_strain = np.concatenate(([left_strain], strain[inside], [right_strain]))
    return ArrayWaveformSource(
        sample_times, windowed_strain.real, -windowed_strain.imag
    )


def dense_tdi_chunks(source, label, duration=DURATION, chunk=16384):
    size = round(duration / DT)
    output = {name: disk_array(f"{label}_{name}", size) for name in ("hp", "hc", "A", "E", "T")}
    detector = SpaceDetector("LISA", backend="pycbc")
    overlap = 256
    for start in range(0, size, chunk):
        stop = min(size, start + chunk)
        padded = np.arange(start - overlap, stop + overlap, dtype=float) * DT
        hp, hc = source.polarizations(padded)
        hp = TimeSeries(hp, delta_t=DT, epoch=padded[0])
        hc = TimeSeries(hc, delta_t=DT, epoch=padded[0])
        channels = detector.project_wave(
            hp, hc, LAMBDA, BETA,
            tdi=TDI_GENERATION, tdi_chan="AET", velocity_order=1,
        )
        center = slice(overlap, overlap + stop - start)
        output["hp"][start:stop], output["hc"][start:stop] = hp[center], hc[center]
        for name in "AET":
            output[name][start:stop] = channels[name][center]
        if start == 0 or stop == size or (start // chunk) % 100 == 0:
            print(f"CCE response: {100 * stop / size:5.1f}%")
        del hp, hc, channels
    for values in output.values():
        values.flush()
    return output

In [7]:
if RUN_CCE:
    import gwsurrogate

    configured_model = os.environ.get("NRHYBSUR_CCE_PATH")
    model_candidates = [
        Path(configured_model).expanduser() if configured_model else None,
        Path("/mnt/d/pyEFPE/.cache/gwsurrogate/NRHybSur3dq8_CCE.h5"),
        Path.home() / ".cache/gwsurrogate/NRHybSur3dq8_CCE.h5",
    ]
    model_path = next(
        (path for path in model_candidates if path is not None and path.is_file()),
        None,
    )
    if model_path is None:
        raise FileNotFoundError(
            "NRHybSur3dq8_CCE.h5 was not found. Set NRHYBSUR_CCE_PATH "
            "to the downloaded model before running the CCE section."
        )
    print("CCE surrogate:", model_path)
    cce_model = gwsurrogate.LoadSurrogate(str(model_path))
    cce_times, cce_strain, cce_memory = cce_waveform(cce_model, cce_parameters)
    del cce_model
    gc.collect()

    cce_source = mission_source(cce_times, cce_strain, cce_memory)
    cce_td = dense_tdi_chunks(cce_source, "cce_smbhb_tdi2")
    cce_curves, cce_snr = summarize_dense(cce_td)
    print("SNR:", cce_snr)

    center = len(cce_td["A"]) // 2
    half_width = round(30 * DAY / DT)
    view = slice(center - half_width, center + half_width)
    days = (np.arange(view.start, view.stop) - center) * DT / DAY
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(days, cce_td["hp"][view], lw=0.7, label=r"$h_+$")
    axes[1].plot(days, cce_td["A"][view], lw=0.7, label="TDI-2.0 A")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    axes[1].set_xlabel("time from merger [day]")
    plt.show()

    plot_lisa_bands(
        cce_curves, cce_snr, "NRHybSur3dq8_CCE SMBHB",
        [(1e-5, 1e-3), (1e-3, 1e-2), (1e-2, 0.1)],
    )


## Memory FFT without periodic-tiling leakage

A plain FFT treats the final memory plateau as adjacent to the initial
baseline and therefore inserts an artificial reverse step at the boundary.
`foutstep.stepdft.syss_dft` subtracts a fitted sigmoid, transforms the
endpoint-compatible residual, and restores the sigmoid analytically.

Here the isolated memory is sampled every 20 s before calling the package
implementation. That is sufficient for the slowly varying memory spectrum
and cuts the two-year transform to about 3.2 million samples. No local copy
of the SySS algorithm is maintained in this notebook.

In [8]:
def crossing_location_width(times, memory):
    values = np.asarray(memory.real)
    progress = (values - values[0]) / (values[-1] - values[0])
    crossings = [times[np.argmin(np.abs(progress - level))] for level in (0.1, 0.5, 0.9)]
    width = (crossings[2] - crossings[0]) / (2 * np.arctanh(0.8))
    return crossings[1], max(width, 20.0)


if RUN_CCE:
    from foutstep.stepdft import syss_dft

    memory_dt = 20.0
    mission_time = np.arange(0.0, DURATION, memory_dt)
    relative_time = mission_time - DURATION / 2
    memory_plus = np.interp(
        relative_time, cce_times, cce_memory.real,
        left=cce_memory[0].real, right=cce_memory[-1].real,
    )
    location, width = crossing_location_width(cce_times, cce_memory)
    location += DURATION / 2

    bare_f = np.fft.rfftfreq(len(memory_plus), memory_dt)
    bare_h = memory_dt * np.fft.rfft(memory_plus)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        syss_f, syss_h = syss_dft(
            mission_time, memory_plus, memory_dt, location, width
        )
    positive = (syss_f > 0) & (syss_f <= 0.1)
    jump = abs(memory_plus[-1] - memory_plus[0])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.loglog(bare_f[1:], 2 * bare_f[1:] * np.abs(bare_h[1:]), label="plain periodic FFT")
    ax.loglog(syss_f[positive], 2 * syss_f[positive] * np.abs(syss_h[positive]), label="gw-foutstep SySS")
    guide_f = np.geomspace(1e-9, 0.1, 600)
    ax.loglog(guide_f, np.full_like(guide_f, jump / np.pi), ls=":", label=r"ideal step $\Delta h/\pi$")
    ax.axvline(1 / DURATION, color="0.5", ls="--", label=r"$1/T_{obs}$")
    ax.set_xlim(1e-9, 0.1)
    ax.set_xlabel("frequency [Hz]")
    ax.set_ylabel(r"memory characteristic strain $2f|\tilde h|$")
    ax.grid(which="both", alpha=0.25)
    ax.legend()
    plt.show()

## Interpretation

`rho_sky_average` and `rho_full_tdi` answer different detector questions and
need not agree for one fixed sky position. Both use the standard
`pycbc.filter.sigma`; what differs is the waveform and PSD supplied to it.
The former uses an averaged strain sensitivity. The latter uses this
source's moving-constellation TDI-2 A/E/T response and an equal-static-arm
analytic channel PSD. That pairing is diagnostic, not yet a production
likelihood.

The Bandopadhyay–Moore/Yorsh GW150914-like injection with reported coherent
SNR 38 is at 50 Mpc and uses different frequency, eccentricity, orientation,
sky and merger-time choices. This notebook uses the GWOSC distance of
420 Mpc, so 38 is not an expected regression value here.

The CCE memory curve below $1/T_{obs}$ is an analytic step guide rather than
measured Fourier bins. Converting it to PTA timing residuals or a PTA SNR
remains a separate detector calculation.
